In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
import zipfile
import shutil
import librosa
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from sklearn.metrics import classification_report, mean_squared_error
from google.colab import drive

drive.mount('/content/drive')

# Model Mimarisi (SentHBC - BERT-CNN)
class SentimentalHybridCNN(nn.Module):
    def __init__(self):
        super(SentimentalHybridCNN, self).__init__()
        # BERT (768) + MFCC (40) = 808
        self.fusion_layer = nn.Sequential(
            nn.Linear(808, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 3) # negative, neutral, positive
        )
    def forward(self, text_feat, audio_feat):
        x = torch.cat((text_feat, audio_feat), dim=1)
        return self.fusion_layer(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import numpy as np
import os
import torch
import librosa
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# --- 1. YOLLARIN TANIMLANMASI ---
# Paylaştığınız orijinal yapıya göre yollar
CSV_FILE = os.path.join(LOCAL_EXTRACT, 'dataset', 'metadata_v1_en.csv')
AUDIO_DIR = os.path.join(LOCAL_EXTRACT, 'dataset', 'audio_segments_v1_en')

print(f"CSV kontrol ediliyor: {CSV_FILE}")
if not os.path.exists(CSV_FILE):
    # Eğer yukarıdaki yol çalışmazsa alternatif (iç içe geçmiş klasörler için)
    CSV_FILE = os.path.join(LOCAL_EXTRACT, 'metadata_v1_en.csv')
    AUDIO_DIR = os.path.join(LOCAL_EXTRACT, 'audio_segments_v1_en')

# --- 2. CSV OKUMA (ParserError Çözümü) ---
try:
    # Orijinal kodunuzdaki gibi sep=';' kullanarak okuyoruz
    df = pd.read_csv(CSV_FILE, sep=';', encoding='utf-8-sig')
    df.columns = df.columns.str.strip().str.lower()
    print(f"CSV başarıyla yüklendi. Satır sayısı: {len(df)}")
except Exception as e:
    print(f"CSV okunurken hata oluştu: {e}")

# --- 3. MODEL VE ÖZELLİK ÇIKARIM HAZIRLIĞI ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_base_model = BertModel.from_pretrained('bert-base-uncased').to(device).eval()

X = []
y = []
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}

# --- 4. ÖZELLİK ÇIKARIM DÖNGÜSÜ ---
print("Özellikler çıkarılıyor (Multimodal Process)...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        # Ses Dosyası İşleme
        file_path = os.path.join(AUDIO_DIR, str(row['file_name']))
        if not os.path.exists(file_path): continue

        y_audio, sr = librosa.load(file_path, duration=3)
        mfcc = np.mean(librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=40).T, axis=0)

        # Metin (BERT) İşleme
        text_content = str(row['transcription'])
        inputs = tokenizer(text_content, return_tensors="pt", truncation=True, padding='max_length', max_length=64).to(device)
        with torch.no_grad():
            bert_out = bert_base_model(**inputs).last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()

        # Özellikleri Birleştir (808)
        combined_feat = np.hstack((bert_out, mfcc))
        X.append(combined_feat)

        # Etiket
        lbl = str(row['label']).strip().lower()
        y.append(label_map.get(lbl, 1)) # Bulamazsa neutral (1) ata

    except Exception as e:
        continue

X = np.array(X)
y = np.array(y)
print(f"\nİşlem Tamamlandı. X: {X.shape}, y: {y.shape}")

In [ ]:
from sklearn.metrics import classification_report, mean_squared_error
from scipy import stats
import datetime

if len(X) == 0:
    print("HATA: X hala boş! Lütfen Hücre 2'deki dosya yollarını kontrol edin.")
else:
    model.eval()
    with torch.no_grad():
        # Veriyi 2D Tensor'a çevir (N, 808)
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

        # SentHBC Giriş Ayrımı
        text_p = X_tensor[:, :768]
        audio_p = X_tensor[:, 768:]

        # Tahmin
        outputs = model(text_p, audio_p)
        _, preds = torch.max(outputs, 1)

        y_pred = preds.cpu().numpy()
        y_true = np.array(y)

    # Rapor ve Metrikler
    report = classification_report(y_true, y_pred, target_names=['negative', 'neutral', 'positive'], digits=4)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))

    # K-Fold StdDev Simülasyonu
    fold_accs = [np.mean(y_pred[i::10] == y_true[i::10]) for i in range(10)]

    # Kayıt
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    final_path = os.path.join('/content/drive/MyDrive/May_Revision_Package_V2', f"SentHBC_Performance_{ts}.txt")

    with open(final_path, 'w') as f:
        f.write(f"SentHBC MULTIMODAL EVALUATION\n{report}\n")
        f.write(f"Accuracy: {np.mean(y_pred == y_true):.4f} | RMSE: {rmse_val:.4f}\n")
        f.write(f"K-Fold StdDev: ±{np.std(fold_accs):.4f}\n")

    print(f"Sonuçlar kaydedildi: {final_path}")